# Inflation in Germany as an additional data source

**Purpose:** download the monthly consumer price index for Germany, turn it into inflation rates, save it as a CSV file in this folder, and show how it is joined to the daily sales **without data leakage**.

**Source:** Eurostat, dataset `prc_hicp_midx`: Harmonised Index of Consumer Prices (HICP), monthly, index 2015 = 100, Germany. Two series:
- `CP00`: all items (general inflation)
- `CP0111`: bread and cereals (prices of the bakery's own product category)

**Period:** January 2012 to July 2019. This is wider than the sales data (07/2013 – 07/2018) on purpose: the year-on-year rate for 2013 needs the 2012 values, the one-month delay needs the month before the first sales day, and the Kaggle test year runs until 07/2019.

## 1. Download from Eurostat

In [1]:
import io

import pandas as pd
import requests

EUROSTAT_URL = (
    "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/"
    "prc_hicp_midx/M.I15.CP00+CP0111.DE?startPeriod=2012-01&endPeriod=2019-07&format=SDMX-CSV"
)

response = requests.get(EUROSTAT_URL, timeout=60, headers={"User-Agent": "opencampus-course-project"})
response.raise_for_status()
raw = pd.read_csv(io.StringIO(response.text))
print(f"Rows downloaded: {len(raw)}")
raw[["coicop", "geo", "unit", "TIME_PERIOD", "OBS_VALUE"]].head()

Rows downloaded: 182


,coicop,geo,unit,TIME_PERIOD,OBS_VALUE
0,CP00,DE,I15,2012-01,95.7
1,CP00,DE,I15,2012-02,96.5
2,CP00,DE,I15,2012-03,96.9
3,CP00,DE,I15,2012-04,97.0
4,CP00,DE,I15,2012-05,96.8


## 2. One row per month

We turn the long table into one row per month with one column per series. Then we compute the **inflation rate**: the change of the index compared with the same month one year earlier, in percent (year-on-year rate).

In [2]:
inflation = (
    raw.pivot(index="TIME_PERIOD", columns="coicop", values="OBS_VALUE")
       .rename(columns={"CP00": "hicp_all_items", "CP0111": "hicp_bread_cereals"})
       .rename_axis(index="month", columns=None)
       .reset_index()
)
inflation["month"] = pd.PeriodIndex(inflation["month"], freq="M")
inflation = inflation.sort_values("month").reset_index(drop=True)

inflation["inflation_all_items_pct"] = (inflation["hicp_all_items"].pct_change(12) * 100).round(2)
inflation["inflation_bread_cereals_pct"] = (inflation["hicp_bread_cereals"].pct_change(12) * 100).round(2)
inflation.tail()

,month,hicp_all_items,hicp_bread_cereals,inflation_all_items_pct,inflation_bread_cereals_pct
86,2019-03,104.4,104.5,1.36,2.45
87,2019-04,105.4,104.8,2.13,2.44
88,2019-05,105.7,104.8,1.34,2.34
89,2019-06,106.0,105.1,1.53,2.44
90,2019-07,106.4,105.3,1.14,2.83


## 3. Plausibility checks

- Every month from 01/2012 to 07/2019 occurs exactly once (91 months), no gaps.
- The index is 2015 = 100, so the average of 2015 must be close to 100.
- German inflation in these years was low, roughly between 0 and 3 percent per year.

In [3]:
expected_months = pd.period_range("2012-01", "2019-07", freq="M")
assert list(inflation["month"]) == list(expected_months)
assert inflation[["hicp_all_items", "hicp_bread_cereals"]].notna().all().all()

mean_2015 = inflation[inflation["month"].dt.year == 2015][["hicp_all_items", "hicp_bread_cereals"]].mean()
print("Average index in 2015 (should be about 100):")
print(mean_2015.round(2))

print("\nAverage inflation rate per year, in percent:")
inflation.groupby(inflation["month"].dt.year)[["inflation_all_items_pct", "inflation_bread_cereals_pct"]].mean().round(2)

Average index in 2015 (should be about 100):
hicp_all_items         99.99
hicp_bread_cereals    100.00
dtype: float64

Average inflation rate per year, in percent:


,inflation_all_items_pct,inflation_bread_cereals_pct
month,,
2012,NaN,NaN
2013,1.61,2.43
2014,0.77,1.35
2015,0.68,1.48
2016,0.37,0.60
2017,1.70,0.42
2018,1.93,1.57
2019,1.55,2.36


## 4. Save

The file is saved next to this notebook. Other notebooks read it from here; the raw bakery data stays unchanged.

In [4]:
inflation.to_csv("inflation_germany_monthly.csv", index=False)
print(f"Saved {len(inflation)} months to inflation_germany_monthly.csv")

Saved 91 months to inflation_germany_monthly.csv


## 5. How to join it to the daily sales without leakage

The forecast for tomorrow is made at closing time. The inflation figure of the **current** month is not yet published at that moment, so a sales day may only use the value of an **earlier** month. The delay is set in `LAG_MONTHS` (see the open question on the canvas; 1 month is the current proposal).

**Check:** the join must not change the number of rows of the sales data (9,334).

In [5]:
LAG_MONTHS = 1  # proposal, to be confirmed by the team (canvas block 6)

SALES_URL = "https://raw.githubusercontent.com/opencampus-sh/intro-to-data-science-and-ml/main/data/bakery/umsatzdaten_gekuerzt.csv"
sales = pd.read_csv(SALES_URL, parse_dates=["Datum"])

sales["inflation_month"] = sales["Datum"].dt.to_period("M") - LAG_MONTHS
joined = sales.merge(
    inflation.rename(columns={"month": "inflation_month"}),
    on="inflation_month",
    how="left",
)

assert len(joined) == len(sales) == 9334
assert joined["hicp_all_items"].notna().all()
joined[["Datum", "Warengruppe", "Umsatz", "inflation_month", "inflation_all_items_pct", "inflation_bread_cereals_pct"]].head()

,Datum,Warengruppe,Umsatz,inflation_month,inflation_all_items_pct,inflation_bread_cereals_pct
0,2013-07-01,1,148.828353,2013-06,1.97,2.42
1,2013-07-02,1,159.793757,2013-06,1.97,2.42
2,2013-07-03,1,111.885594,2013-06,1.97,2.42
3,2013-07-04,1,168.864941,2013-06,1.97,2.42
4,2013-07-05,1,171.280754,2013-06,1.97,2.42


## What inflation can and cannot do in this project

- It changes only once a month and slowly. Over five years it mostly behaves like a **slow trend**, so a model can easily confuse it with any other long-term change (new products, more customers, price increases of this bakery).
- Turnover is measured in money. Inflation can therefore also be used to express turnover in prices of one base year (**real turnover**). Whether we use it as a feature, for that conversion, or both is a team decision.
- The data has no prices of this bakery. The bread-and-cereals index shows national price changes, not the branch's own prices.